# Compare four LLM APIs

How do different models answer the same question?
We will ask about a Swiss voting initiative, first in English and then in German.

You will write four short API functions. The prompts and comparison code are provided.
Later, you will add a source text and try the OpenAI Decisions API.

Work through one section at a time. Complete each `TODO` when you reach it.
You can check your work in `llm_intro_solution.ipynb`.

We use the historical initiative **“Für Ehe und Familie – gegen die Heiratsstrafe”**,
which was voted on **28 February 2016**. Keep this date when asking about it to avoid
confusing it with later proposals about marriage taxation.

## Get ready

Use Python 3.12. Open a terminal in this repository and create a virtual environment:

```sh
python -m venv .venv
```

Activate it using the command for your system.

**Windows PowerShell**

```powershell
.venv\Scripts\Activate.ps1
```

**macOS or Linux**

```sh
source .venv/bin/activate
```

Then install the packages:

```sh
python -m pip install -r requirements.txt
```

### Add your keys and open the notebook

Copy `.env.example` to a new file named `.env`.
Fill in the four API keys. Keep this file private.
The model names are already supplied.

Start JupyterLab:

```sh
python -m jupyterlab
```

Open `llm_intro.ipynb` and select the `.venv` Python kernel.
Run the cells in order.

### Load the libraries and settings

Run the next two cells. They load the API keys and model names from `.env`.
You do not need to change this code.

Check that each provider prints `key configured: True` before continuing.

In [1]:
import os                         #Zugriff auf Betriebssystemfunktionen und Umgebungsvariablen (Environment Variables)

import requests                   #Senden von HTTP-Anfragen an externe APIs
import anthropic                  #Zugriff auf die Claude-Modelle von Anthropic
from dotenv import load_dotenv    #Laden von Umgebungsvariablen, bspw. API-Schlüsseln, aus einer .env-Datei
from google import genai          #Zugriff auf die Gemini-Modelle von Google
from openai import OpenAI         #Zugriff auf die KI-Modelle von OpenAI

In [2]:
load_dotenv(".env")                                                                         #Umgebungsvariablen aus der .env-Datei laden

google_api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")                 #Google API-Schlüssel laden, alternativ GEMINI_API_KEY verwenden
openai_api_key = os.getenv("OPENAI_API_KEY")                                                #OpenAI API-Schlüssel laden
deepinfra_api_key = os.getenv("DEEPINFRA_API_KEY")                                          #DeepInfra API-Schlüssel laden
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")                                          #Anthropic API-Schlüssel laden

google_model = os.getenv("GOOGLE_MODEL", "gemini-3.8-flash")                                #Google-Modell laden oder angegebenes Standardmodell verwenden
openai_model = os.getenv("OPENAI_MODEL", "gpt-6-luna")                                      #OpenAI-Modell laden oder angegebenes Standardmodell verwenden
deepinfra_model = os.getenv("DEEPINFRA_MODEL", "deepseek-ai/DeepSeek-V4.1-Flash")           #DeepInfra-Modell laden oder Standardmodell verwenden
anthropic_model = os.getenv("ANTHROPIC_MODEL", "claude-haiku-5-5")                          #Anthropic-Modell laden oder angegebenes Standardmodell verwenden

# Prüfen, ob die API-Schlüssel vorhanden sind, ohne diese offenzulegen
for provider, key in [("Google", google_api_key), ("OpenAI", openai_api_key),
                      ("DeepInfra", deepinfra_api_key), ("Anthropic", anthropic_api_key)]:  #Alle Anbieter und ihre API-Schlüssel durchlaufen
    print(provider, "key configured:", bool(key))                                           #Anbieter und Status ausgeben, True bei vorhandenem Schlüssel, sonst False

Google key configured: True
OpenAI key configured: True
DeepInfra key configured: True
Anthropic key configured: True


## The question we will ask

Read the two prompts below. They ask the same question in English and German.
Run the cell to save them as `prompt_en` and `prompt_de`.

We will reuse these variables so every provider receives exactly the same text
within each language.

In [3]:
prompt_en = """Explain the Swiss popular initiative "Für Ehe und Familie – gegen die
Heiratsstrafe", voted on 28 February 2016, to someone unfamiliar with it.
What did it propose about taxes, social insurance and the definition of marriage?
Answer neutrally in English, using at most 120 words. Say if you do not know a detail."""

prompt_de = """Erkläre die Schweizer Volksinitiative «Für Ehe und Familie – gegen die
Heiratsstrafe», über die am 28. Februar 2016 abgestimmt wurde, für jemanden ohne Vorwissen.
Was schlug sie zu Steuern, Sozialversicherungen und zur Definition der Ehe vor?
Antworte neutral auf Deutsch mit höchstens 120 Wörtern. Sage, wenn du ein Detail nicht weisst."""

## Connect to the four providers

Each function follows the same pattern:

1. Create a client using the API key.
2. Send the prompt to the selected model.
3. Return the answer as a string.

`return` lets us save the answer and compare it later. The comparison code will print it.

### 1. Google Gemini

Pass the prompt as `contents`. The answer is in `response.text`.

[API examples](https://ai.google.dev/api/generate-content)

Complete `ask_google(prompt)` below.

1. Create the client with `genai.Client(api_key=google_api_key)`.
2. Call `client.models.generate_content(...)` with `model=google_model`
   and `contents=prompt`. Save the result as `response`.
3. Return `response.text`.

In [4]:
#Ausgangslage
def ask_google(prompt):
    # TODO: send prompt to this provider and return the answer as a string.
    raise NotImplementedError("Implement ask_google")

In [5]:
def ask_google(prompt):                                 #Funktion definieren, die einen Prompt als Eingabeparameter erhält
    client = genai.Client(api_key=google_api_key)       #API-Client erstellen und mit dem API-Schlüssel authentifizieren
    response = client.models.generate_content(          #Anfrage an die Google Gemini API senden und Antwort speichern
        model=google_model,                             #Zu verwendendes KI-Modell festlegen
        contents=prompt                                 #Prompt als Eingabe an das Modell übergeben
    )
    return response.text                                #Generierten Antworttext aus dem Antwortobjekt extrahieren und zurückgeben

### 2. OpenAI

Here we use the Responses API. Pass the prompt as `input`.
The answer is in `response.output_text`.

[API examples](https://developers.openai.com/api/docs/guides/text)

Complete `ask_openai(prompt)` below.

1. Create the client with `OpenAI(api_key=openai_api_key)`.
2. Call `client.responses.create(...)` with `model=openai_model`
   and `input=prompt`. Save the result as `response`.
3. Return `response.output_text`.

In [6]:
#Ausgangslage
def ask_openai(prompt):
    # TODO: send prompt to this provider and return the answer as a string.
    raise NotImplementedError("Implement ask_openai")

In [7]:
def ask_openai(prompt):                         #Funktion definieren, die einen Prompt als Eingabeparameter erhält
    client = OpenAI(api_key=openai_api_key)     #Client-Objekt aus der OpenAI-Bibliothek erstellen und API-Schlüssel zur Authentifizierung übergeben
    response = client.responses.create(         #Anfrage über die OpenAI API senden und das Antwortobjekt in response speichern
        model=openai_model,                     #Zu verwendendes KI-Modell festlegen
        input=prompt                            #Prompt als Eingabe an das Modell übergeben
    )
    return response.output_text                 #Generierten Antworttext über die bereits in der OpenAI-Bibliothek definierte Eigenschaft output_text auslesen und zurückgeben

### 3. DeepInfra

DeepInfra works with the OpenAI Python package, using its own key and API address.
This call sends the prompt as a user message.

[API examples](https://docs.deepinfra.com/chat/overview)

Complete `ask_deepinfra(prompt)` below.

1. Create an `OpenAI` client using `api_key=deepinfra_api_key` and
   `base_url="https://api.deepinfra.com/v1/openai"`.
2. Call `client.chat.completions.create(...)` with `model=deepinfra_model`
   and the user message below. Save the result as `response`.

```python
messages=[{"role": "user", "content": prompt}]
```

3. Return `response.choices[0].message.content`. This is the first answer's text.

In [8]:
#Ausgangslage
def ask_deepinfra(prompt):
    # TODO: send prompt to this provider and return the answer as a string.
    raise NotImplementedError("Implement ask_deepinfra")

In [9]:
def ask_deepinfra(prompt):                                      #Funktion definieren, die einen Prompt als Eingabeparameter erhält
    client = OpenAI(                                            #Client-Objekt aus der OpenAI-Bibliothek erstellen
        api_key=deepinfra_api_key,                              #DeepInfra API-Schlüssel zur Authentifizierung übergeben
        base_url="https://api.deepinfra.com/v1/openai"          #Zieladresse der DeepInfra API festlegen
    )
    response = client.chat.completions.create(                  #Anfrage an die DeepInfra API senden und Antwortobjekt speichern
        model=deepinfra_model,                                  #Zu verwendendes KI-Modell festlegen
        messages=[{"role": "user", "content": prompt}]          #Prompt als Nachricht mit der Benutzerrolle übergeben
    )
    return response.choices[0].message.content                  #Generierten Antworttext der ersten Antwort extrahieren und zurückgeben

### 4. Anthropic

Anthropic also receives a user message. Its answer comes back in several blocks.
Keep the blocks whose type is `"text"` and join their text into one string.

[API examples](https://platform.claude.com/docs/en/api/messages/create)

Complete `ask_anthropic(prompt)` below.

1. Create the client with `anthropic.Anthropic(api_key=anthropic_api_key)`.
2. Call `client.messages.create(...)` with `model=anthropic_model`,
   `max_tokens=4096` and the user message below. Save the result as `response`.

```python
messages=[{"role": "user", "content": prompt}]
```

3. Collect `block.text` from the blocks in `response.content`
   where `block.type == "text"`.
4. Join those strings with `"\n"` and return the result.

`max_tokens=4096` sets the output budget. It leaves room for the model's
internal reasoning as well as its short answer.

In [10]:
#Ausgangslage
def ask_anthropic(prompt):
    # TODO: send prompt to this provider and return the answer as a string.
    raise NotImplementedError("Implement ask_anthropic")

In [11]:
def ask_anthropic(prompt):                                      #Funktion definieren, die einen Prompt als Eingabeparameter erhält
    client = anthropic.Anthropic(api_key=anthropic_api_key)     #Client-Objekt erstellen und API-Schlüssel zur Authentifizierung übergeben
    response = client.messages.create(                          #Anfrage an die Anthropic API senden und Antwortobjekt speichern
        model=anthropic_model,                                  #Zu verwendendes KI-Modell festlegen
        max_tokens=4096,                                        #Maximale Anzahl generierter Tokens festlegen
        messages=[{"role": "user", "content": prompt}]          #Prompt als Nachricht mit der Benutzerrolle übergeben
    )
    return "\n".join(                                           #Alle extrahierten Antworttexte mit Zeilenumbrüchen verbinden und zurückgeben
        block.text for block in response.content                #Alle Inhaltsblöcke durchlaufen und deren Text extrahieren
        if block.type == "text"                                 #Nur Inhaltsblöcke vom Typ text berücksichtigen
    )

## Compare the answers

The function below is provided. It sends `prompt` to each of the four API functions
and prints their answers.

It takes only one argument: the prompt text. Each API function already selects its model.
For example, `compare_models(prompt_en)` uses the English prompt.

Run this cell to define the function. It will make requests when we call it below.

In [12]:
def compare_models(prompt):                             #Funktion definieren, die denselben Prompt an mehrere KI-Modelle sendet
    answers = {}                                        #Leeres Dictionary zum Speichern der Modellantworten erstellen

    print("\nGoogle Gemini")                            #Modellnamen mit vorangestelltem Zeilenumbruch ausgeben
    answers["Google Gemini"] = ask_google(prompt)       #Prompt an Google Gemini senden und Antwort im Dictionary speichern
    print(answers["Google Gemini"])                     #Gespeicherte Antwort ausgeben

    print("\nOpenAI")                                   #Modellnamen mit vorangestelltem Zeilenumbruch ausgeben
    answers["OpenAI"] = ask_openai(prompt)              #Prompt an OpenAI senden und Antwort im Dictionary speichern
    print(answers["OpenAI"])                            #Gespeicherte Antwort ausgeben

    print("\nDeepInfra")                                #Modellnamen mit vorangestelltem Zeilenumbruch ausgeben
    answers["DeepInfra"] = ask_deepinfra(prompt)        #Prompt an DeepInfra senden und Antwort im Dictionary speichern
    print(answers["DeepInfra"])                         #Gespeicherte Antwort ausgeben

    print("\nAnthropic")                                #Modellnamen mit vorangestelltem Zeilenumbruch ausgeben
    answers["Anthropic"] = ask_anthropic(prompt)        #Prompt an Anthropic senden und Antwort im Dictionary speichern
    print(answers["Anthropic"])                         #Gespeicherte Antwort ausgeben

    return answers                                      #Vollständiges Dictionary mit allen vier Modellantworten zurückgeben

### A. Ask without a source text

Start with English. Run the next cell and read the four answers before continuing.

Each comparison sends four API requests. Re-running a cell sends them again
and may incur charges. The full notebook makes 26 requests.

In [13]:
answers_en = compare_models(prompt_en)  #Funktion mit dem englischen Prompt aufrufen, Antworten aller vier KI-Modelle ausgeben und im Dictionary answers_en speichern


Google Gemini
According to official Swiss government documentation, the popular initiative "Für Ehe und Familie – gegen die Heiratsstrafe" (For Marriage and Family – Against the Marriage Penalty) was launched by the Christian Democratic People's Party (CVP) and put to a national vote on 28 February 2016.

The initiative proposed amending Article 14 of the Swiss Federal Constitution to address three main areas:

*   **Taxes:** It sought to eliminate the "marriage penalty," stipulating that married couples should not be taxed more heavily than unmarried cohabiting couples with equivalent joint incomes.
*   **Social Insurance:** It required that married couples not face disadvantages compared to unmarried couples regarding social security benefits, such as caps on old-age and survivors' insurance (AHV) pensions.
*   **Definition of Marriage:** It proposed defining marriage constitutionally as the legally regulated union between a man and a woman. 

Proponents argued that the measure woul

Now run the German version. Do the models describe the same proposals in both languages?

MINL: Die Modelle beschreiben in beiden Sprachen grundsätzlich dieselben Vorschläge bezüglich Steuern, Sozialversicherungen und der Definition der Ehe. Allerdings unterscheiden sich einzelne Details, bspw. erwähnt Anthropic in der deutschen Antwort das Splitting-Verfahren, in der englischen Antwort hingegen Kinder als Bestandteil der Ehedefinition. Diese Unterschiede können durch sprachabhängige Trainingsdaten, unterschiedliche Formulierungen der Prompts und die probabilistische Textgenerierung der LLMs entstehen. Die Kernaussagen sind somit sprachübergreifend konsistent, während zusätzliche Informationen und deren Genauigkeit variieren

In [14]:
answers_de = compare_models(prompt_de)      #Funktion mit dem deutschen Prompt aufrufen, Antworten aller vier KI-Modelle ausgeben und im Dictionary answers_en speichern


Google Gemini
Gemäss den amtlichen Erläuterungen der Schweizerischen Bundeskanzlei verlangte die von der damaligen CVP (heute «Die Mitte») lancierte Initiative Folgendes:

* **Steuern und Sozialversicherungen:** Die Initiative forderte, dass verheiratete Paare gegenüber unverheirateten Paaren nicht benachteiligt werden dürfen. Nach Ansicht der Befürworter sollte damit die sogenannte «Heiratsstrafe» beseitigt werden, bei der Verheiratete durch die gemeinsame Veranlagung oft in höhere Steuerstufen fielen oder bei der AHV eine gedeckelte Maximalrente erhielten.
* **Definition der Ehe:** Der Text sah vor, die Ehe in der Bundesverfassung ausdrücklich als dauerhafte und gesetzlich geregelte Lebensgemeinschaft von Mann und Frau zu definieren.

Kritiker bemängelten insbesondere diese Definition, da sie die Öffnung der Ehe für gleichgeschlechtliche Paare («Ehe für alle») verfassungsrechtlich blockiert hätte. 

Die Vorlage wurde im Februar 2016 mit 50,8 % Nein-Stimmen knapp abgelehnt. Das Bunde

### B. Give the models a source text

Read the two snippets below. They summarize the proposal voted on in 2016.
They do not describe current law or give voting results.

These are teaching summaries based on the
[Federal Council's description](https://www.admin.ch/de/fur-ehe-und-familie).

In [15]:
source_text_en = """The 2016 initiative sought to prevent disadvantages for married
couples in taxation and social insurance compared with other living arrangements.
It proposed treating spouses as one economic unit for tax purposes and defining
marriage as a lasting union between a man and a woman. The Federal Council and
Parliament recommended rejecting the initiative."""

source_text_de = """Die Initiative von 2016 wollte Benachteiligungen von Ehepaaren
bei Steuern und Sozialversicherungen gegenüber anderen Lebensformen verhindern.
Sie wollte Ehepaare steuerlich als eine Wirtschaftsgemeinschaft behandeln und die
Ehe als dauerhafte Verbindung von Mann und Frau definieren. Bundesrat und Parlament
empfahlen, die Initiative abzulehnen."""

We will keep the original question and add the matching snippet.
We also ask the models to use only the supplied text.

Run the next cell and read the completed English prompt that it prints.

In [16]:
prompt_with_text_en = (                                                                         #Neuen englischen Prompt aus mehreren Strings zusammensetzen
    f"{prompt_en}\n\n"                                                                          #Ursprünglichen englischen Prompt einfügen, gefolgt von zwei Zeilenumbrüchen
    "Use only the source text below. If a detail is missing, say it is not in the text.\n\n"    #Anweisung zur ausschliesslichen Verwendung des Quelltexts ergänzen
    f"Source text:\n{source_text_en}"                                                           #Englischen Quelltext einfügen
)

prompt_with_text_de = (                                                                         #Neuen deutschen Prompt aus mehreren Strings zusammensetzen
    f"{prompt_de}\n\n"                                                                          #Ursprünglichen deutschen Prompt einfügen
    "Verwende nur den folgenden Quelltext. Wenn ein Detail fehlt, sage, dass es nicht im Text steht.\n\n"   #Anweisung zur Verwendung des Quelltexts ergänzen
    f"Quelltext:\n{source_text_de}"                                                                         #Deutschen Quelltext einfügen
)

print(prompt_with_text_en)                                                                      #Vollständig zusammengesetzten englischen Prompt ausgeben

Explain the Swiss popular initiative "Für Ehe und Familie – gegen die
Heiratsstrafe", voted on 28 February 2016, to someone unfamiliar with it.
What did it propose about taxes, social insurance and the definition of marriage?
Answer neutrally in English, using at most 120 words. Say if you do not know a detail.

Use only the source text below. If a detail is missing, say it is not in the text.

Source text:
The 2016 initiative sought to prevent disadvantages for married
couples in taxation and social insurance compared with other living arrangements.
It proposed treating spouses as one economic unit for tax purposes and defining
marriage as a lasting union between a man and a woman. The Federal Council and
Parliament recommended rejecting the initiative.


Run the English comparison. Which details in the answers can you find in the snippet?

In [17]:
answers_with_text_en = compare_models(prompt_with_text_en)      #Englischen Prompt inkl. Quelltext an alle vier Modelle senden, Antworten ausgeben und im Dictionary speichern


Google Gemini
Based on the provided text, the 2016 initiative proposed the following:

* **Taxes:** It sought to prevent tax disadvantages for married couples compared with other living arrangements and proposed treating spouses as one economic unit for tax purposes.
* **Social insurance:** It proposed preventing disadvantages for married couples in social insurance compared with other living arrangements; further specific details are not in the text.
* **Definition of marriage:** It proposed defining marriage as a lasting union between a man and a woman. 

The text also notes that the Federal Council and Parliament recommended rejecting the initiative. Any other background details are not in the text.

OpenAI
The initiative aimed to prevent married couples from being disadvantaged compared with people in other living arrangements, both in taxation and in social insurance. It proposed treating spouses as a single economic unit for tax purposes and defining marriage as a lasting union 

Repeat with the German prompt and snippet.

In [18]:
answers_with_text_de = compare_models(prompt_with_text_de)  #Deutschen Prompt inkl. Quelltext an alle vier Modelle senden, Antworten ausgeben und im Dictionary speichern


Google Gemini
Die Schweizer Volksinitiative von 2016 wollte verhindern, dass Ehepaare im Vergleich zu anderen Lebensformen benachteiligt werden. 

Sie schlug dazu Folgendes vor:
* **Steuern:** Ehepaare sollten steuerlich als eine Wirtschaftsgemeinschaft behandelt werden, um Benachteiligungen zu verhindern.
* **Sozialversicherungen:** Auch hier sollten Benachteiligungen von Ehepaaren gegenüber anderen Lebensformen verhindert werden. Weitere konkrete Details dazu stehen nicht im Text.
* **Definition der Ehe:** Die Ehe sollte als dauerhafte Verbindung von Mann und Frau definiert werden.

Bundesrat und Parlament empfahlen, die Initiative abzulehnen.

OpenAI
Die Initiative «Für Ehe und Familie – gegen die Heiratsstrafe» kam am 28. Februar 2016 zur Abstimmung. Sie wollte Nachteile von Ehepaaren gegenüber anderen Lebensformen bei Steuern und Sozialversicherungen verhindern. Bei den Steuern sollten Ehepaare als eine Wirtschaftsgemeinschaft behandelt werden. Zur konkreten Ausgestaltung bei den

MINL:
- Alle vier Modelle geben die wesentlichen Informationen aus dem Quelltext wieder: die Vermeidung von Benachteiligungen bei Steuern und Sozialversicherungen, die Besteuerung als Wirtschaftsgemeinschaft, die Definition der Ehe als dauerhafte Verbindung zwischen Mann und Frau sowie die Ablehnungsempfehlung von Bundesrat und Parlament. Fehlende Details zur konkreten Umsetzung werden als solche gekennzeichnet. Das von Anthropic erwähnte Abstimmungsdatum stammt aus dem ursprünglichen Prompt und nicht aus dem Quelltext
- Durch die Bereitstellung des Quelltexts sind die Antworten deutlich konsistenter und enthalten weniger unbelegte Aussagen
Durch die Bereitstellung des Quelltexts sind die Antworten deutlich konsistenter und enthalten weniger unbelegte Aussagen, da alle Modelle auf derselben Informationsgrundlage antworten (Grounding), weniger auf ihr erlerntes Wissen aus den Trainingsdaten zurückgreifen müssen und explizit angewiesen werden, fehlende Informationen nicht zu ergänzen. Dadurch wird das Risiko von Halluzinationen reduziert

### How this prepares us for RAG

Here, we chose a passage and put it into the prompt ourselves.

Later, **retrieval-augmented generation (RAG)** will find relevant passages in documents
and add them to the prompt. The model then answers using that information.
We have just practised this last step.

### C. Ask about missing information

The snippets do not give the percentage of people who voted yes.
What should a model say if it must answer using only this text?

Read the two prompts below, then run the cell to save them.

In [19]:
missing_prompt_en = (
    "What percentage voted yes on the initiative on 28 February 2016? "
    "Answer in English using only the source text. "
    "If the answer is missing, say it is not in the text.\n\n"
    f"Source text:\n{source_text_en}"
)

missing_prompt_de = (
    "Wie hoch war der Ja-Stimmenanteil zur Initiative am 28. Februar 2016? "
    "Antworte auf Deutsch und verwende nur den Quelltext. "
    "Wenn die Antwort fehlt, sage, dass sie nicht im Text steht.\n\n"
    f"Quelltext:\n{source_text_de}"
)

Try the English question. Check whether each answer acknowledges that the percentage is missing.

In [20]:
missing_answers_en = compare_models(missing_prompt_en)  #Englischen Prompt mit einer Frage zu fehlenden Informationen an alle vier Modelle senden, Antworten ausgeben und im Dictionary speichern


Google Gemini
It is not in the text.

OpenAI
The percentage is not in the text.

DeepInfra
It is not in the text.

Anthropic
The text does not say what percentage voted yes on the initiative on 28 February 2016. It only describes the initiative's aims and notes that the Federal Council and Parliament recommended rejecting it.


Now try it in German. Does any model supply a figure despite the instruction to use only the text?

In [21]:
missing_answers_de = compare_models(missing_prompt_de)


Google Gemini
Diese Information steht nicht im Quelltext.

OpenAI
Der Ja-Stimmenanteil steht nicht im Text.

DeepInfra
Die Antwort steht nicht im Text.

Anthropic
Der Ja-Stimmenanteil steht nicht im Text.


## Pause and compare

Write a short note in each column. Mention differences between English and German
where you notice them.

| Provider | Without text | With text | Missing information |
| --- | --- | --- | --- |
| Google Gemini | | | |
| OpenAI | | | |
| DeepInfra | | | |
| Anthropic | | | |

Discuss these questions:

- What changed when you supplied the text?
- Did changing the language change any facts?
- Did the models admit when information was missing?
- Which answers would you check before sharing them with someone else?

You can read a saved answer again without making another request:

```python
answers_with_text_en["DeepInfra"]
```

If a call fails, check that provider's API key, credits and model access.
Then re-run the comparison cell.

#### Pause and compare

| Provider | Without text | With text | Missing information |
|---|---|---|---|
| Google Gemini | Gleiche Kernaussagen in beiden Sprachen. Englische Antwort erwähnt zusätzlich die AHV-Plafonierung von 150% und die Aufhebung des Abstimmungsergebnisses | Konsistente Antworten in beiden Sprachen, basierend auf dem Quelltext | Erkennt in beiden Sprachen, dass der Ja-Stimmenanteil fehlt |
| OpenAI | Weitgehend konsistente Antworten. Deutsche Antwort erläutert zusätzlich die AHV-Plafonierung | Beschränkt sich weitgehend auf den Quelltext und kennzeichnet fehlende Details | Erkennt in beiden Sprachen die fehlende Information |
| DeepInfra | Unterschiede bei den Aussagen zur AHV und zur steuerlichen Entlastung. Englische Antwort nennt zusätzlich den Nein-Stimmenanteil | Konsistente Antworten ohne unbelegte Angaben zu Steuern oder AHV | Erkennt in beiden Sprachen die fehlende Information |
| Anthropic | Englische Antwort erwähnt Kinder als Bestandteil der Ehedefinition, deutsche Antwort das Splitting-Verfahren | Konsistente Kernaussagen und Kennzeichnung fehlender Details | Erkennt in beiden Sprachen die fehlende Information. Englische Antwort enthält eine zusätzliche Begründung |

#### Discuss these questions

##### What changed when you supplied the text?

- Die Antworten wurden konsistenter und enthielten weniger unbelegte Aussagen, da alle Modelle dieselbe Informationsgrundlage verwendeten (Grounding)
- Die Modelle mussten weniger auf ihr erlerntes Wissen aus den Trainingsdaten zurückgreifen und wurden explizit angewiesen, fehlende Informationen nicht zu ergänzen. Dadurch wurde das Risiko von Halluzinationen reduziert

##### Did changing the language change any facts?

- Ohne Quelltext unterschieden sich einzelne behauptete Fakten und Zusatzinformationen zwischen den englischen und deutschen Antworten
- Diese Unterschiede können durch sprachabhängige Trainingsdaten, unterschiedliche Formulierungen der Prompts und die probabilistische Textgenerierung der LLMs entstehen
- Mit Quelltext waren die Kernaussagen weitgehend identisch

##### Did the models admit when information was missing?

- Alle vier Modelle erkannten in beiden Sprachen, dass der Ja-Stimmenanteil nicht im Quelltext enthalten war
- Kein Modell ergänzte einen unbelegten Prozentwert. Alle Modelle hielten sich somit bei dieser Frage an die Anweisung

##### Which answers would you check before sharing them with someone else?

- Insbesondere die Antworten ohne Quelltext sollten überprüft werden, da sie zusätzliche und potenziell falsche Informationen enthalten
- Bspw. sollten die Aussagen von Anthropic zum Splitting-Verfahren und zu Kindern als Bestandteil der Ehedefinition sowie die Aussagen von DeepInfra zur AHV überprüft werden
- Auch Antworten mit Quelltext sollten überprüft werden, da Grounding das Risiko von Halluzinationen reduziert, aber keine vollständige Fehlerfreiheit garantiert

## One more API: OpenAI Decisions

So far, we have asked models to write answers. Now we will ask two yes/no questions
about a snippet:

1. Does it say spouses would be treated as one economic unit for taxes?
2. Does it give the percentage who voted yes?

The [Decisions API](https://developers.openai.com/api/docs/guides/decisions)
can return a probability for each question. A value near 1 means the model thinks
the answer is yes; a value near 0 means it thinks the answer is no.

This kind of check is called a `predicate`. Read the two questions below and run the cell.
The same questions will be used for both snippets.

In [22]:
decision_questions = [                              #Liste mit zwei Entscheidungsfragen erstellen
    {                                               #Dictionary für die erste Entscheidungsfrage
        "type": "predicate",                        #Fragetyp als Prädikat definieren, das mit True oder False beantwortet werden soll
        "name": "tax_unit",                         #Eindeutigen Namen für die Frage zur Wirtschaftsgemeinschaft festlegen
        "instructions": (                           #Fragestellung für das KI-Modell definieren
            "Does the text say the initiative proposed treating spouses "
            "as one economic unit for tax purposes?"
        ),
    },
    {                                               #Dictionary für die zweite Entscheidungsfrage
        "type": "predicate",                        #Fragetyp erneut als Prädikat definieren
        "name": "vote_percentage",                  #Eindeutigen Namen für die Frage zum Ja-Stimmenanteil festlegen
        "instructions": (                           #Fragestellung inkl. Einschränkung auf den Quelltext definieren
            "Does the text give the percentage who voted yes? "
            "Use only the supplied text."
        ),
    },
]

### Send the snippet and the questions

We use `requests.post` to send a JSON request directly to the API.
This also works with OpenAI SDK versions that do not yet include Decisions.
The request uses the same OpenAI key as before.

For this endpoint, use `gpt-6-luna`. The function returns a list of answer dictionaries,
so its result has a different format from the four text-generation functions.

Complete `ask_openai_decisions(source_text)` below.

Send `requests.post(...)` with these arguments:

- URL: `"https://api.openai.com/v1/decisions"`
- `headers`: `{"Authorization": f"Bearer {openai_api_key}"}`
- `timeout`: `60` seconds
- `json`: the dictionary shown below

```python
{
    "model": "gpt-6-luna",
    "input": source_text,
    "questions": decision_questions,
}
```

Save the result as `response`. Then:

1. Call `response.raise_for_status()` to report an unsuccessful request.
2. Return `response.json()["answers"]` to get the list of answers.

In [23]:
#Ausgangslage
def ask_openai_decisions(source_text):
    # TODO: send the snippet and both predicate questions to the Decisions API.
    # Return the list of answers from the JSON response.
    raise NotImplementedError("Implement ask_openai_decisions")

In [ ]:
def ask_openai_decisions(source_text):                              #Funktion definieren, die einen Quelltext als Eingabeparameter erhält
    response = requests.post(                                       #HTTP-POST-Anfrage an die Decisions API senden und Antwortobjekt speichern
        "https://api.openai.com/v1/decisions",                      #Zieladresse der API festlegen
        headers={"Authorization": f"Bearer {openai_api_key}"},      #API-Schlüssel zur Authentifizierung übergeben
        timeout=60,                                                 #Timeout von 60 Sekunden festlegen
        json={                                                      #Anfragedaten als Dictionary übergeben, das automatisch in JSON umgewandelt wird
            "model": "gpt-6-luna",                                  #Zu verwendendes KI-Modell festlegen
            "input": source_text,                                   #Quelltext als Informationsgrundlage übergeben
            "questions": decision_questions                         #Zuvor definierte Liste mit beiden Entscheidungsfragen übergeben
        }
    )
    response.raise_for_status()                                     #Bei einem HTTP-Fehler eine Exception auslösen
    return response.json()["answers"]                               #JSON-Antwort in eine Python-Datenstruktur umwandeln und Liste der Antworten zurückgeben

### Read the results

Before running the English snippet, predict which question should get a high probability.
Then run the cell. Each printed dictionary is one answer:

- `name` identifies the question.
- `type` should be `predicate`.
- `probability` is the model's estimate.

If the type is `refusal`, the API declined the check and did not give a probability.

In [ ]:
decision_answers_en = ask_openai_decisions(source_text_en)      #Englischen Quelltext an die Decisions API senden und zurückgegebene Liste mit Antworten speichern
for answer in decision_answers_en:                              #Alle Antworten in der Liste nacheinander durchlaufen
    print(answer)                                               #Jeweilige Antwort ausgeben

{'type': 'predicate', 'name': 'tax_unit', 'probability': 1.0}
{'type': 'predicate', 'name': 'vote_percentage', 'probability': 1.0}


Repeat with the German snippet. Do the results agree with your reading of both texts?
A probability is still a model estimate, so check it against the snippet.

In [ ]:
decision_answers_de = ask_openai_decisions(source_text_de)      #Deutschen Quelltext an die Decisions API senden und zurückgegebene Liste mit Antworten speichern
for answer in decision_answers_de:                              #Alle Antworten in der Liste nacheinander durchlaufen
    print(answer)                                               #Jeweilige Antwort ausgeben

{'type': 'predicate', 'name': 'tax_unit', 'probability': 1.0}
{'type': 'predicate', 'name': 'vote_percentage', 'probability': 0.91}


MINL
- **tax_unit:** Das Modell liefert für beide Sprachen eine Wahrscheinlichkeit von 1.0 (100%). Dies ist korrekt, da die Besteuerung als Wirtschaftsgemeinschaft in beiden Quelltexten erwähnt wird
- **vote_percentage:** Das Modell liefert für den englischen Quelltext eine Wahrscheinlichkeit von 1.0 (100%) und für den deutschen Quelltext von 0.91 (91%). Beide Ergebnisse sind falsch, da der Ja-Stimmenanteil in keinem der beiden Quelltexte enthalten ist. Die erwartete Wahrscheinlichkeit wäre nahe 0
- **Sprachabhängigkeit:** Die Wahl der Sprache beeinflusst die geschätzte Wahrscheinlichkeit, obwohl beide Quelltexte dieselben Informationen enthalten
- **Fazit:** Eine hohe Wahrscheinlichkeit bedeutet nicht automatisch, dass eine Antwort korrekt ist. Auch strukturierte Antworten mit Wahrscheinlichkeitswerten müssen anhand des Quelltexts überprüft werden